### Finlora Project

##### Problem Statement
- Finlora’s current transaction monitoring framework, built on static rule-based thresholds, is no longer sufficient to address the complexity and scale of modern fraud. As transaction volumes increase across mobile, web, API, and card channels, fraud patterns have evolved to become more subtle and diverse. The existing system struggles to balance detection accuracy with customer experience, resulting in both missed fraud and excessive false positives.

#### Core Challenges
- Operational inefficiency: Analysts are overwhelmed by a high volume of low-value alerts, leaving limited capacity to investigate genuinely suspicious activity.
- Data underutilization: Current rules evaluate transactions in isolation, without leveraging behavioral context such as account history, device usage, or spending baselines.
- Lack of prioritization: All flagged transactions are treated equally, with no scoring mechanism to differentiate high-risk cases from routine anomalies.

##### Business Impact
- Efficiency loss: Manual review scales linearly with transaction volume, creating unsustainable operational overhead.
- Financial risk: Missed fraud leads to direct losses and chargebacks, while false positives increase support costs and drive customer churn.
- Customer trust erosion: Legitimate transactions—such as payroll, wires, or P2P transfers—are delayed or blocked, undermining confidence in Finlora as a reliable financial hub.

##### Strategic Need
- To remain competitive and secure, Finlora requires a smarter, adaptive monitoring system that can weigh multiple signals simultaneously, segment risk by account type and channel, and prioritize alerts based on actual risk. Without this evolution, the widening gap between static rules and modern fraud will translate into material financial losses and a friction-heavy customer experience.


In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

#### Loading the Data

In [40]:
Accounts = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\AMDARI\FINLORA PROJECT\DATA\RAW\finlora_accounts.csv")
Transactions = pd.read_csv(r"C:\Users\HP\OneDrive\Documents\AMDARI\FINLORA PROJECT\DATA\RAW\finlora_transactions.csv")

In [41]:
Accounts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   account_id                   7200 non-null   object 
 1   account_holder_name          7200 non-null   object 
 2   account_type                 7200 non-null   object 
 3   home_country                 7200 non-null   object 
 4   currency                     7200 non-null   object 
 5   kyc_tier                     7200 non-null   object 
 6   account_created_date         7200 non-null   object 
 7   personal_spend_baseline_usd  7200 non-null   float64
dtypes: float64(1), object(7)
memory usage: 450.1+ KB


In [42]:
Transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   transaction_id              126000 non-null  object 
 1   account_id                  126000 non-null  object 
 2   account_type                126000 non-null  object 
 3   kyc_tier                    126000 non-null  object 
 4   timestamp                   126000 non-null  object 
 5   day_of_week                 126000 non-null  object 
 6   hour_of_day                 126000 non-null  int64  
 7   description                 126000 non-null  object 
 8   merchant_name               97390 non-null   object 
 9   merchant_category           126000 non-null  object 
 10  channel                     126000 non-null  object 
 11  amount                      126000 non-null  float64
 12  currency                    126000 non-null  object 
 13  amount_to_avg_

#### Data Merging

In [43]:
merged_df = pd.merge(Accounts, Transactions, how='inner', on='account_id')

In [44]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 31 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   account_id                   126000 non-null  object 
 1   account_holder_name          126000 non-null  object 
 2   account_type_x               126000 non-null  object 
 3   home_country_x               126000 non-null  object 
 4   currency_x                   126000 non-null  object 
 5   kyc_tier_x                   126000 non-null  object 
 6   account_created_date         126000 non-null  object 
 7   personal_spend_baseline_usd  126000 non-null  float64
 8   transaction_id               126000 non-null  object 
 9   account_type_y               126000 non-null  object 
 10  kyc_tier_y                   126000 non-null  object 
 11  timestamp                    126000 non-null  object 
 12  day_of_week                  126000 non-null  object 
 13 

In [45]:
merged_df.nunique()

account_id                       7144
account_holder_name              1000
account_type_x                      2
home_country_x                      5
currency_x                          4
kyc_tier_x                          3
account_created_date             1441
personal_spend_baseline_usd      6262
transaction_id                 126000
account_type_y                      2
kyc_tier_y                          3
timestamp                      112316
day_of_week                         7
hour_of_day                        24
description                      8644
merchant_name                      63
merchant_category                  15
channel                             6
amount                          90856
currency_y                          4
amount_to_avg_ratio              5205
avg_transaction_amount_30d      74185
transaction_velocity_1h             4
transaction_country                 5
home_country_y                      5
is_cross_border                     2
device_id   

In [46]:
merged_df.head()

,account_id,account_holder_name,account_type_x,home_country_x,currency_x,kyc_tier_x,account_created_date,personal_spend_baseline_usd,transaction_id,account_type_y,...,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country_y,is_cross_border,device_id,is_new_device,account_age_days,status,is_fraud
0,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,8/28/2022,63.23,FLR250216186265,Individual,...,9143.33,0,NG,NG,0,NaN,NaN,903,Completed,0
1,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,8/28/2022,63.23,FLR250423143305,Individual,...,9143.33,0,NG,NG,0,NaN,NaN,969,Declined,0
2,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,8/28/2022,63.23,FLR250424154897,Individual,...,9143.33,0,NG,NG,0,DEV-9055235108,0.0,970,Completed,0
3,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,8/28/2022,63.23,FLR250428101772,Individual,...,14891.16,0,NG,NG,0,NaN,NaN,974,Completed,0
4,FLR-ACC-100000,Amaka Okafor,Individual,NG,NGN,Tier3_Enhanced,8/28/2022,63.23,FLR250628102827,Individual,...,49609.59,0,NG,NG,0,DEV-9055235108,0.0,1035,Declined,0


#### Checking for missing values

In [47]:
merged_df.isnull().sum()

account_id                         0
account_holder_name                0
account_type_x                     0
home_country_x                     0
currency_x                         0
kyc_tier_x                         0
account_created_date               0
personal_spend_baseline_usd        0
transaction_id                     0
account_type_y                     0
kyc_tier_y                         0
timestamp                          0
day_of_week                        0
hour_of_day                        0
description                        0
merchant_name                  28610
merchant_category                  0
channel                            0
amount                             0
currency_y                         0
amount_to_avg_ratio                0
avg_transaction_amount_30d         0
transaction_velocity_1h            0
transaction_country                0
home_country_y                     0
is_cross_border                    0
device_id                      14334
i

In [48]:
merged_df.isnull().mean()*100

account_id                      0.000000
account_holder_name             0.000000
account_type_x                  0.000000
home_country_x                  0.000000
currency_x                      0.000000
kyc_tier_x                      0.000000
account_created_date            0.000000
personal_spend_baseline_usd     0.000000
transaction_id                  0.000000
account_type_y                  0.000000
kyc_tier_y                      0.000000
timestamp                       0.000000
day_of_week                     0.000000
hour_of_day                     0.000000
description                     0.000000
merchant_name                  22.706349
merchant_category               0.000000
channel                         0.000000
amount                          0.000000
currency_y                      0.000000
amount_to_avg_ratio             0.000000
avg_transaction_amount_30d      0.000000
transaction_velocity_1h         0.000000
transaction_country             0.000000
home_country_y  

#### Handling missing values and removal of duplicate columns from data merger

In [49]:
# Impute missing binary values with mode
merged_df['is_new_device'] = merged_df['is_new_device'].fillna(merged_df['is_new_device'].mode()[0])

# Drop device_id column and other Id columns
merged_df.drop(columns=['device_id'],inplace=True)
merged_df.drop(columns=['account_id'],inplace=True)
merged_df.drop(columns=['transaction_id'],inplace=True)

# Drop Name Column
merged_df.drop(columns=['account_holder_name'],inplace=True)

# Drop duplicate columns
merged_df.drop(columns=['account_type_x'],inplace=True)
merged_df.drop(columns=['home_country_x'],inplace=True)
merged_df.drop(columns=['currency_x'],inplace=True)
merged_df.drop(columns=['kyc_tier_x'],inplace=True)


# Replacing missing value with a new category
# merged_df['merchant_name'] = merged_df['merchant_name'].fillna('unknown')
# Function to apply conditional logic
def fill_merchant(row):
    if pd.isna(row['merchant_name']):
        if row['description'] and 'NIP' in row['description']:
            return 'P2P'
        elif row['description'] and 'Salary' in row['description']:
            return 'Payroll'
        else:
            return 'Unknown'
    else:
        return row['merchant_name']

# Apply the function row by row
merged_df['merchant_name'] = merged_df.apply(fill_merchant, axis=1)

In [50]:
merged_df.isnull().sum()

account_created_date           0
personal_spend_baseline_usd    0
account_type_y                 0
kyc_tier_y                     0
timestamp                      0
day_of_week                    0
hour_of_day                    0
description                    0
merchant_name                  0
merchant_category              0
channel                        0
amount                         0
currency_y                     0
amount_to_avg_ratio            0
avg_transaction_amount_30d     0
transaction_velocity_1h        0
transaction_country            0
home_country_y                 0
is_cross_border                0
is_new_device                  0
account_age_days               0
status                         0
is_fraud                       0
dtype: int64

In [51]:
merged_df.head()

,account_created_date,personal_spend_baseline_usd,account_type_y,kyc_tier_y,timestamp,day_of_week,hour_of_day,description,merchant_name,merchant_category,...,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,transaction_country,home_country_y,is_cross_border,is_new_device,account_age_days,status,is_fraud
0,8/28/2022,63.23,Individual,Tier3_Enhanced,2/16/2025 14:55,Sunday,14,CNP PURCHASE - BOLT,Bolt,Travel,...,9.21,9143.33,0,NG,NG,0,0.0,903,Completed,0
1,8/28/2022,63.23,Individual,Tier3_Enhanced,4/23/2025 13:35,Wednesday,13,WEB PURCHASE - SLACK,Slack,Subscription/SaaS,...,1.00,9143.33,0,NG,NG,0,0.0,969,Declined,0
2,8/28/2022,63.23,Individual,Tier3_Enhanced,4/24/2025 12:30,Thursday,12,MOBILE PURCHASE - JUSTRITE SUPERSTORE,Justrite Superstore,Groceries,...,2.26,9143.33,0,NG,NG,0,0.0,970,Completed,0
3,8/28/2022,63.23,Individual,Tier3_Enhanced,4/28/2025 14:41,Monday,14,CNP PURCHASE - ADOBE CREATIVE CLOUD,Adobe Creative Cloud,Subscription/SaaS,...,0.21,14891.16,0,NG,NG,0,0.0,974,Completed,0
4,8/28/2022,63.23,Individual,Tier3_Enhanced,6/28/2025 16:15,Saturday,16,MOBILE PURCHASE - APPLE STORE,Apple Store,Electronics,...,1.97,49609.59,0,NG,NG,0,0.0,1035,Declined,0


In [52]:
# Check for duplicate rows
merged_df.duplicated().sum()

np.int64(0)

In [53]:
merged_df.shape

(126000, 23)

In [54]:
merged_df.describe()

,personal_spend_baseline_usd,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,is_new_device,account_age_days,is_fraud
count,126000.000000,126000.000000,1.260000e+05,126000.000000,1.260000e+05,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000
mean,292.277286,13.542111,7.964918e+05,7.921400,7.675995e+05,0.011429,0.020198,0.030913,354.721992,0.026603
std,631.504181,5.260369,4.054387e+06,153.566706,3.290377e+06,0.110323,0.140679,0.173082,331.327033,0.160921
min,3.770000,0.000000,-1.834602e+06,0.000000,-1.669153e+06,0.000000,0.000000,0.000000,0.000000,0.000000
25%,44.510000,9.000000,6.310500e+01,0.250000,9.242750e+01,0.000000,0.000000,0.000000,35.000000,0.000000
50%,92.190000,14.000000,5.942240e+03,0.980000,5.942240e+03,0.000000,0.000000,0.000000,289.000000,0.000000
75%,296.230000,18.000000,1.208470e+05,1.830000,1.575701e+05,0.000000,0.000000,0.000000,577.000000,0.000000
max,10469.650000,23.000000,2.691614e+08,31762.690000,1.733621e+08,3.000000,1.000000,1.000000,1682.000000,1.000000


#### Transaction Amounts
- Mean transaction amount is very high (~796,000), but the median is only ~5,942.
- This huge gap suggests extreme outliers (some transactions in the hundreds of millions).
- The huge figure difference may also be a result of the different amount currency.
- Negative values in amount and avg_transaction_amount_30d indicate possible data errors or chargebacks/refunds.
  
#### Spending Baseline
- Median baseline spend is ~92 USD, but the mean is ~292 USD.
- Again, skewed by high spenders.
- Maximum baseline is over 10,000 USD, showing some accounts with very high normal spending.
  
#### Time of Day
- Transactions are spread across all hours (0–23).
- Median is 14 (2 PM), suggesting peak activity in the afternoon.
  
#### Fraud Indicator
- Only ~2.6% of transactions are labeled as fraud.
- Fraud cases are rare compared to legitimate ones, which is typical in real-world datasets.
  
#### Cross-Border & New Device
- Very low mean values (~2–3%), meaning most transactions are domestic and from known devices.
  
#### Account Age
- Median account age is ~289 days.
- Fraud tends to be more common in newer accounts (closer to 0 days).
  
#### Transaction Velocity
- Mean is very low (~0.01), but max is 3.
- Indicates some accounts make multiple transactions within an hour, which could be suspicious.

In [57]:
merged_df.currency_y.value_counts()

currency_y
NGN    64185
USD    24900
GBP    18966
EUR    17949
Name: count, dtype: int64

In [55]:
merged_df.select_dtypes('number').corr()

,personal_spend_baseline_usd,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,is_new_device,account_age_days,is_fraud
personal_spend_baseline_usd,1.000000,0.001252,0.328820,0.069049,0.387650,0.013617,0.004652,0.002790,0.024612,0.024793
hour_of_day,0.001252,1.000000,-0.007345,-0.004077,-0.005624,0.010686,0.002224,-0.000109,-0.002936,0.003477
amount,0.328820,-0.007345,1.000000,0.226321,0.527842,0.018889,0.004647,0.003453,0.004093,0.063006
amount_to_avg_ratio,0.069049,-0.004077,0.226321,1.000000,-0.010459,-0.004399,-0.001411,-0.000264,0.007634,0.155231
avg_transaction_amount_30d,0.387650,-0.005624,0.527842,-0.010459,1.000000,0.023282,0.001579,0.001506,0.003296,-0.001848
transaction_velocity_1h,0.013617,0.010686,0.018889,-0.004399,0.023282,1.000000,-0.008226,0.006852,-0.007592,0.577002
is_cross_border,0.004652,0.002224,0.004647,-0.001411,0.001579,-0.008226,1.000000,0.004344,-0.001109,0.006764
is_new_device,0.002790,-0.000109,0.003453,-0.000264,0.001506,0.006852,0.004344,1.000000,0.017073,0.035442
account_age_days,0.024612,-0.002936,0.004093,0.007634,0.003296,-0.007592,-0.001109,0.017073,1.000000,-0.013157
is_fraud,0.024793,0.003477,0.063006,0.155231,-0.001848,0.577002,0.006764,0.035442,-0.013157,1.000000


##### Strong Relationships
- Transaction velocity vs fraud
- Correlation = 0.577 → This is the strongest signal. Accounts making multiple transactions in a short time are highly associated with fraud.

##### Amount-to-average ratio vs fraud
- Correlation = 0.155 → Transactions far above or below the user’s average spending are moderately linked to fraud.

##### Transaction amount vs fraud
- Correlation = 0.063 → Larger transaction amounts show a weak but positive association with fraud.
